In [1]:
import numpy as np
import fitz
import regex as re
from langchain_text_splitters import RecursiveCharacterTextSplitter
#from qdrant_client.models import Distance, VectorParams
##from qdrant_client import QdrantClient
#from qdrant_client.models import PointStruct
from google import genai
from google.genai import types
import faiss
import json
import pathlib




/Users/Dennis/python_projects/chatbot_for_CV/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:

file = "CV_Dennis_Gloukhman_eng.pdf"
collection_name = 'CV_chunks'

client = genai.Client()

wd = pathlib.Path().resolve()._str
data_folder =wd +'/backend/data/'

def extract_words_from_pdf(file):
    prefix = "backend/data/"
    file_path = prefix+file

    doc = fitz.open(file_path)
    pages =[]
    for num, page in enumerate(doc):
        text = ""
        meta_info ={}
        meta_info ['source'] = file
        meta_info['page_nr'] = num+1
        text += page.get_text() 
        meta_info ['text'] = text
        pages.append(meta_info)


    regex = re.compile('[^a-zA-Z0-9\\.\\n\\/ \\@]')
    
    for page in pages:
         page['text'] = regex.sub('', page['text'] )
         page['text']  = str(page['text'] .replace('\n', ' ') )
    return pages


def create_chunks(input_list):
    chunker = RecursiveCharacterTextSplitter(chunk_size=300,
    chunk_overlap=25,
    length_function=len,
    is_separator_regex=True,)
    chunks = []
    for input in input_list:
        docs = chunker.split_text(input['text'])
        for num , chunk in enumerate(docs):
            chunk = chunk.replace('\n\n|\n|\r', ' ') 
            chunk = str(re.sub(r'\s+', ' ', chunk.strip()))
            chunk = re.sub(r"[ \t]+", " ", chunk)
            chunk = re.sub(r"\n{3,}", "\n\n", chunk) 
            res ={
                'page_nr': input['page_nr'],
                'source' :  input['source'],
                'chunk_nr': num+1,
                'text' : chunk
            } 
            chunks.append(res)
    return chunks


# Generate embedding for a task's query. Use your correct task here:
def prepare_query(query):

    return f"task: RETRIEVAL_DOCUMENT | query: {query}"

# Generate embedding for document of an asymmetric retrieval task:
def prepare_document(content, title=None):
    if title is None:
        title = "none"
    return f"title: {title} | text: {content}"

def embed_documents( documents):
    embeddings = []
    for document in documents:
        result = client.models.embed_content(
            model='gemini-embedding-2',
            contents=document['text'],
        )
        embedding = result.embeddings[0].values
        embeddings.append(embedding)

    return np.asarray(
        embeddings,
        dtype=np.float32,
    )


def embed_query(query):
    prepared_query = prepare_query(query)
    result = client.models.embed_content(
        model='gemini-embedding-2',
        contents=prepared_query,
    )
    embedding = result.embeddings[0].values

    return np.asarray(
        embedding,
        dtype=np.float32,
    )

def search(query, k=3):
    query_embedding = embed_query(query)

    scores, indices = index.search(
        np.array([query_embedding]),
        k
    )

    return [
        chunks[i]
        for i in indices[0]
    ]
# def create_embedding(docs):
#     client = genai.Client()
#     embeddings   = client.models.embed_content(
#     model="gemini-embedding-001",
#     contents=[test_string],
#     config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT")
# )
#     embeddings = model.encode(docs)
#     vector_size = embeddings[0].size
#     return zip(embeddings, docs), vector_size

# def insert_qdrant(vec, chunk, id):
#     operation_info = client.upsert(
#         collection_name="CV_chunks",
#         wait=True,
#         points=[
#         PointStruct(id=id, vector=vec, payload= {'chunk':chunk}),
#     ],)
#     return print(operation_info)

# def create_qdrantdb (collection, vec_size):
#     client = QdrantClient(url="http://localhost:6333")
#     if client.collection_exists(
#     collection_name=collection) != True:
#         client.create_collection(
#    collection_name=collection,
#     vectors_config=VectorParams(size=vec_size, distance=Distance.COSINE),)
#         return client  


In [ ]:

extracted_words = extract_words_from_pdf(file)
chunks = create_chunks(extracted_words)
embeddings = embed_documents(chunks)

# Create FAISS index
index = faiss.IndexFlatIP(embeddings.shape[1])

# Add document embeddings
index.add(embeddings)


faiss.write_index(index, data_folder+"cv.index")



with open(data_folder+"chunks.json", "w") as f:
    json.dump(chunks, f)


#index = faiss.read_index("cv.index")



# id = 0
# for vec, chunk in embeddings:
#     insert_qdrant(vec, chunk, id)
#     id += 1


In [ ]:
import faiss
import json

index = faiss.read_index(data_folder+"cv.index")
with open(data_folder+"chunks.json") as f:
    chunks = json.load(f)

query = "What experience does Dennis have with Python?"
#query = "Does Dennis have experience in consulting?"

query_embedding = embed_query(
    query
)

scores, indices = index.search(
    np.array([query_embedding]),
    3
)


results = [chunks[i] for i in indices[0]]


#for result in results:
#    print(result)

In [ ]:
from backend.app.prompt import SYSTEM_PROMPT


#client = genai.Client()


def create_prompt(question,context):
    cont = ''
    for num, i in enumerate(context):
        cont += f'''Context {num+1}
        Source file: {i['source']}
        Page in Source: {i['page_nr']}
        Text of context: {i['text']}\n'''


    return f"""
CONTEXT:
{cont}

USER QUESTION:
{question}
"""

prompt = create_prompt (query, results)

# response = client.models.generate_content(
#     model="gemini-3.8-flash",
#     contents=prompt,
#     config={
#             "system_instruction": SYSTEM_PROMPT,
#         },
#     )

In [13]:
print(response.text)

Dennis is skilled in Python and used it for data migration and transformation during his internship at KPMG Austria (Pages 1 and 2).


In [20]:

query2 = 'Whats the most recent work experienceof Denis?'
query_embedding = embed_query(
    query2
)

scores, indices = index.search(
    np.array([query_embedding]),
    7
)

results = [chunks[i] for i in indices[0]]

print(results)

prompt =create_prompt(query2,results)
response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents=prompt,
    config={
            "system_instruction": SYSTEM_PROMPT,
        },
    )

print(response.text)

[{'page_nr': 1, 'source': 'CV_Dennis_Gloukhman_eng.pdf', 'chunk_nr': 1, 'text': 'Dennis Gloukhman Tel Aviv Israel 49 176 9149 4485 click dennisgloukhman@hotmail.de LinkedIn click Profile Business Data Analyst with an interdisciplinary background in Business Analytics Psychology and Finance. Expe rienced in requirements engineering big data analytics ETL agile development'}, {'page_nr': 2, 'source': 'CV_Dennis_Gloukhman_eng.pdf', 'chunk_nr': 13, 'text': 'to errors. Solution Use web scraping to automate and accelerate task. Results Saved time and created reusable solution to facilitate data collection. Tel Aviv 12 July 2026 Dennis Gloukhman'}, {'page_nr': 1, 'source': 'CV_Dennis_Gloukhman_eng.pdf', 'chunk_nr': 10, 'text': 'classification model to reduce manual article screening. Achieved 99 recall while reducing workload by 54. 10/2017 11/2020 B.Sc. Business Administration Vienna University of Economics and Business Focus Finance Entrepreneurship Innovation Semester abroad Copenhagen Bus

In [22]:
response.contents

AttributeError: 'GenerateContentResponse' object has no attribute 'contents'

In [17]:
print(response.text)


The information about Dennis's most recent work experience is not available in the provided CV context.


In [ ]:

client = genai.Client()


def create_prompt(
    question: str,
    context: str,
) -> str:

    return f"""
CV CONTEXT:
{context}

USER QUESTION:
{question}
"""




response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents=prompt,
    config={
            "system_instruction": SYSTEM_PROMPT,
        },
    )

In [ ]:
from google import genai

client = genai.Client()
response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Write a story about a magic backpack. Only 2 setences"
)
print(response.text)

Whenever Maya reached into the frayed backpack, its bottomless interior magically produced whatever she needed most at that exact moment. Standing lost in the freezing woods, she unzipped the pocket with shivering hands and pulled out a glowing, palm-sized campfire.


In [ ]:
# test_string = 'what was your master thesis in buisness analytics'
# client = genai.Client()
# test   = client.models.embed_content(
#     model="gemini-embedding-001",
#     contents=[test_string],
#     config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT")
# )


# client_db = QdrantClient(url="http://localhost:6333")


# search_result = client_db.query_points(
#     collection_name=collection_name,
#     query=test.embeddings[0].values,
#     with_payload=True,
#     limit=6
# ).points


#print(search_result)
 

In [8]:

GENERATION_MODEL = "gemini-3.8-flash"
bot = client.models.generate_content(
    model=GENERATION_MODEL,
    contents="Say hello in one sentence."
)

print(bot.text)

Hello, I hope you are having a wonderful day!


In [36]:

search_result = client_db.query_points(
    collection_name=collection_name,
    query=test.embeddings[0].values,
    with_payload=True,
    limit=6
).points


search_result


[ScoredPoint(id=10, version=11, score=0.8346533, payload={'chunk': 'Business School 09/2019 02/2020 Thesis Evaluating the Time Series Momentum Strategy on the German Equity Market 10/2015 10/2018 B.Sc. Psychology University of Vienna Thesis The Link Between Cognitive Ability and Political Attitude in Austria Ushape Skills Languages Hebrew intermediate English'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=6, version=7, score=0.824148, payload={'chunk': 'development of a mobile privacy application. Requirement engineering app testing and bug reporting. Data analysis and reporting. Education 10/2020 03/2026 M.Sc. Business Analytics University of Vienna Focus Banking Finance Big Data Analytics Quantitative Methods Parallel Computing Semester'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=8, version=9, score=0.8069347, payload={'chunk': 'with VR and biofeedback concepts. Investigating feasibility of personalized digital therapy solutions. 10/2020 03/

In [ ]:
client = genai.Client()
result = client.models.embed_content(
    model="gemini-embedding-001",
    contents=chunks,
    config=types.EmbedContentConfig(task_type="SEMANTIC_SIMILARITY")
)

In [ ]:
print(len(result.embeddings[0].values))

TypeError: 'list' object is not callable

In [14]:
len (chunks)

25

In [23]:
from pydantic import BaseModel


In [ ]:
query2

class ChatRequest(BaseModel):
    message : str

mes = ChatRequest (message=query2)

mes.message    

def chat(test):
    request = test
    return  request.message 

chat(mes)

'Whats the most recent work experienceof Denis?'

In [44]:
mes = ChatRequest
mes

__main__.ChatRequest

In [45]:
mes.message

AttributeError: message

In [50]:
def chat(request: ChatRequest):
    return {"answer": f"You asked: {request.message}"}

In [51]:
chat(mes)

{'answer': 'You asked: hkkl'}